# Лабораторная работа № 6. Исследование параметров text-to-image генерации

**Исполнитель:** Яковенко Максим Михайлович
**Дисциплина:** «Искусственный интеллект в креативных технологиях»
**Вариант 2.** Номер варианта N = (номер брифа − 1)·6 + номер фактора = (1 − 1)·6 + 2 = 2:

* бриф 1 — «афиша фестиваля робототехники» (светящийся силуэт механической руки, синий и янтарный, без текста, логотипов и людей);
* фактор 2 — **размер кадра 512×512 → 384×384** (вопрос: пригоден ли уменьшенный кадр для черновиков);
* серия seed 101…105; остальные условия — базовые: структурированный английский запрос, 1 шаг, `guidance_scale = 0,0`, генератор шума на CPU.

**Модели:** `stabilityai/sd-turbo`, ревизия `b261bac6fd2cf515557d5d0707481eafa0485ec2`; метрика соответствия запросу — `openai/clip-vit-base-patch32`, ревизия `3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268`.

**Среда:** Python-ноутбук в Engee (engee.com), рабочая папка `/user/Яковенко_Максим_Михайлович/LR06`. Методические указания рассчитаны на Google Colab с GPU T4; в Engee работа выполняется на GPU, библиотеки преподавателя используются без изменений (`cuda`, `float16`). Отличия от проверенной конфигурации (модель GPU, версии библиотек) фиксируются в разделе 1 и в паспорте; запасной путь для машины без CUDA описан в разделе 3.

**Репозиторий работы:** https://github.com/ykvnkm/ai-in-creative-industries/tree/main/LR06 (гипотеза и допуск записаны в `reports/journal.md` до запуска).

### Структура ноутбука (по плану раздела 5 указаний)

| Шаг плана | Раздел ноутбука |
| --- | --- |
| 1. Подготовка среды | 1–3: пакеты и версии, библиотеки `lab05_lib.py` и `lab06_lib.py`, проверка устройства (запасной путь без CUDA) |
| 2. Гипотеза и допуск | 4 |
| 3. Загрузка модели и CLIP | 5 |
| 4. Режимы A и B | 6 |
| 5. `variant_summary`, таблица метрик | 7 |
| 6. Сравнительный лист seed 101 | 8 (и проверка на одном масштабе — требование варианта 2) |
| 7. Классификация по допуску, 2s против допуска | 9 |
| 8. Намеренная ошибка демо (guidance = 1,0) | 10 |
| — | 11: воспроизведение демонстрационного варианта 1 и сопоставление с указаниями |
| 9. Сохранение результатов, паспорт, вывод | 12–15 |

**Оценка времени на GPU.** На T4 одно изображение 512 за 1 шаг занимает около 0,3 с, поэтому все генерации (серия варианта 2, сравнительный лист, проверка guidance, воспроизведение демо-варианта с 1 и 4 шагами — около 40 изображений) укладываются в 1–2 мин. Основное время — установка недостающих пакетов и первая загрузка весов (≈ 3,3 ГБ): ориентировочно 5–10 мин на весь ноутбук. Если CUDA не окажется и сработает запасной путь на CPU — 15–25 мин.

## 1. Подготовка среды (шаг 1 плана)

Ячейка ниже:

1. создаёт рабочую папку `/user/Яковенко_Максим_Михайлович/LR06` и делает её текущей (если каталога `/user` нет — работает в текущей папке), чтобы `%%writefile` и все артефакты попадали в одно место; создаёт подпапку `artifacts/`;
2. проверяет, какие пакеты уже установлены, и доустанавливает недостающие. `torch` при отсутствии ставится стандартной сборкой с поддержкой CUDA (Engee предоставляет GPU); для `diffusers` и `transformers` сначала пробуются версии из проверенной конфигурации указаний (0.40.0 и 5.16.1), при неудаче — последние доступные. Уже установленные пакеты не трогаются, чтобы не ломать окружение Engee.

`pip` вызывается через `subprocess`, а не `!pip`, чтобы ячейка была обычным Python-кодом и ошибка установки была видна в выводе.

**Версия torch.** В окружении Python Engee может стоять старый `torch`, с которым diffusers 0.40 и transformers 5 не работают (transformers сообщает «PyTorch was not found»). Ячейка проверяет версию по метаданным пакета и при версии ниже 2.4 обновляет `torch` (и `torchvision`, если он установлен) — сборкой с CUDA, если в сеансе есть GPU. После обновления ячейка останавливается: ядро нужно перезапустить и выполнить ноутбук с начала, потому что уже загруженный модуль `torch` заменить без перезапуска нельзя.


In [ ]:
import os, sys, pathlib, subprocess, importlib, time

T_START = time.perf_counter()
WORK = pathlib.Path("/user/Яковенко_Максим_Михайлович/LR06") if pathlib.Path("/user").is_dir() else pathlib.Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
ART = WORK / "artifacts"
ART.mkdir(exist_ok=True)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("Рабочая папка:", WORK)


def have(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print("pip install", " ".join(args), "->", "ok" if r.returncode == 0 else "ошибка\n" + r.stderr[-1500:])
    return r.returncode == 0


# torch проверяется по метаданным пакета, без импорта: старый torch, загруженный в ядро, нельзя заменить на лету.
import re, shutil
from importlib.metadata import version as pkg_version, PackageNotFoundError


def installed_version(pkg):
    try:
        return pkg_version(pkg)
    except PackageNotFoundError:
        return None


def at_least(v, need):
    return v is not None and tuple(int(x) for x in re.findall(r"\d+", v.split("+")[0])[:2]) >= need


TORCH_V = installed_version("torch")
HAS_GPU = shutil.which("nvidia-smi") is not None or os.path.exists("/dev/nvidia0")
print("torch в окружении:", TORCH_V, "| GPU в сеансе:", HAS_GPU)
if not at_least(TORCH_V, (2, 4)):
    # diffusers 0.40 и transformers 5 требуют torch >= 2.4; со старым torch transformers пишет
    # «PyTorch was not found», а diffusers падает при импорте конвейера.
    pkgs = ["torch>=2.4"] + (["torchvision"] if installed_version("torchvision") else [])
    extra = [] if HAS_GPU else ["--index-url", "https://download.pytorch.org/whl/cpu"]
    print("обновляю:", " ".join(pkgs), "(сборка с CUDA)" if HAS_GPU else "(сборка для CPU)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs, *extra], check=True)
    raise SystemExit("torch обновлён до " + str(installed_version("torch")) +
                     ". Перезапустите ядро (Restart) и выполните ноутбук с первой ячейки.")
installed = []
for mod, pinned in [("diffusers", "diffusers==0.40.0"), ("transformers", "transformers==5.16.1")]:
    if not have(mod):
        (pip(pinned) or pip(mod)) and installed.append(mod)
for mod, pkg in [("scipy", "scipy"), ("safetensors", "safetensors"), ("PIL", "Pillow"), ("accelerate", "accelerate")]:
    if not have(mod):
        pip(pkg) and installed.append(pkg)
importlib.invalidate_caches()
print("Доустановлено:", ", ".join(installed) if installed else "ничего, все пакеты уже были")

Фиксируем среду для паспорта: версии Python и библиотек, вычислительное устройство. На GPU выводится имя карты (аналог `nvidia-smi`), на CPU — модель процессора из `/proc/cpuinfo`, число ядер, число потоков `torch` и объём памяти из `/proc/meminfo`. Отдельно печатается выдержка из `pip list` по ключевым пакетам (шаг 1 плана просит версии библиотек).

In [ ]:
import platform
import numpy, scipy, PIL, torch, diffusers, transformers


def proc_field(path, key):
    try:
        for line in open(path, encoding="utf-8"):
            if line.startswith(key):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "н/д"


ENV = dict(python=platform.python_version(), platform=platform.platform(), torch=torch.__version__,
           diffusers=diffusers.__version__, transformers=transformers.__version__, pillow=PIL.__version__,
           numpy=numpy.__version__, scipy=scipy.__version__, cuda=torch.cuda.is_available())
if ENV["cuda"]:
    ENV["device"] = torch.cuda.get_device_name(0)
else:
    ENV["device"] = "CPU: " + (proc_field("/proc/cpuinfo", "model name") if os.path.exists("/proc/cpuinfo") else platform.processor())
ENV["cpu_count"] = os.cpu_count()
ENV["torch_threads"] = torch.get_num_threads()
ENV["mem_total"] = proc_field("/proc/meminfo", "MemTotal")
ENV["mem_available"] = proc_field("/proc/meminfo", "MemAvailable")
for k, v in ENV.items():
    print(f"{k:14s} {v}")

freeze = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True).stdout
keys = ("torch", "diffusers", "transformers", "accelerate", "safetensors", "huggingface", "tokenizers", "scipy", "numpy", "pillow")
print("\npip list (выдержка):")
print("\n".join(l for l in freeze.splitlines() if l.lower().startswith(keys)))

## 2. Учебные библиотеки

По указаниям (раздел 4, п. 3) библиотеки вводятся в блокнот ячейками `%%writefile`. Текст после первой строки каждой ячейки **байт в байт совпадает** с выданными файлами: `lab05_lib.py` из приложения А указаний к ЛР 5 (модель SD Turbo с закреплённой ревизией, функция генерации, брифы) и `lab06_lib.py` из приложения А указаний к ЛР 6 (CLIP, метрики, факторы, `variant_summary`). Файлы в библиотеках не редактируются: всё, что нужно для работы без GPU, сделано отдельно в разделе 3.

Ячейка 1 — `lab05_lib.py`:

In [ ]:
%%writefile lab05_lib.py
# ЛР 5. Запуск открытой модели text-to-image. Библиотека учебных функций.
# Проверено 19.09.2026: Google Colab, GPU T4, Python 3.13, torch 2.11.0+cu128, diffusers 0.40.0, transformers 5.16.1.
# Модель: stabilityai/sd-turbo, ревизия закреплена (см. REVISION). Веса загружаются с Hugging Face без авторизации.
import hashlib, io, time
import numpy as np
import torch
from PIL import Image
from diffusers import AutoPipelineForText2Image
from diffusers.utils import logging as dlog

dlog.set_verbosity_error()  # убирает предупреждения о приведении типов при pipe.to(dtype)

MODEL_ID = "stabilityai/sd-turbo"
REVISION = "b261bac6fd2cf515557d5d0707481eafa0485ec2"

# Пять контекстов (брифов). Во всех запретены текст, логотипы и люди.
BRIEFS = [
    "editorial poster concept for a robotics festival, glowing mechanical arm silhouette, deep blue and amber palette, clean composition, no text, no logo, no people",
    "square podcast cover about the city environment, abstract layered skyline of geometric shapes, teal and orange palette, minimal flat style, no text, no people",
    "poster concept for a science exhibition, abstract constellation of connected nodes, dark background with white and cyan lines, balanced composition, no text, no logo",
    "visual for an ecology lecture series, stylized forest canopy seen from above, layered green shapes, soft morning light, no text, no people",
    "cover art for an electronic music release, brushed metal surface with iridescent reflections, macro photography style, strong contrast, no text, no logo, no people",
]


def load_pipe(dtype=torch.float16):
    """Загружает закреплённую ревизию (файлы fp16) на GPU; dtype задаёт тип вычислений."""
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID, revision=REVISION, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    pipe.set_progress_bar_config(disable=True)
    return pipe.to("cuda", dtype)


def generate(pipe, prompt, seed, steps=1, guidance=0.0, size=512, batch=1, gen_device="cpu"):
    """Возвращает список PIL-изображений. Для каждого элемента пакета свой генератор: seed, seed+1, ..."""
    gens = [torch.Generator(device=gen_device).manual_seed(seed + i) for i in range(batch)]
    return pipe(prompt=[prompt] * batch, num_inference_steps=steps, guidance_scale=guidance,
                height=size, width=size, generator=gens).images


def generate_fixed_noise(pipe, prompt, seed, steps=1, guidance=0.0, size=512):
    """Начальный шум всегда строится в float32 и приводится к типу вычислений: так меняется только точность."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    lat = torch.randn((1, 4, size // 8, size // 8), generator=g, dtype=torch.float32).to("cuda", pipe.unet.dtype)
    return pipe(prompt=[prompt], num_inference_steps=steps, guidance_scale=guidance, height=size, width=size, latents=lat).images[0]


def png_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def sha(data):
    return hashlib.sha256(data).hexdigest()


def to_arr(img):
    return np.asarray(img.convert("RGB"), dtype=np.float64)


def psnr_db(a, b):
    """PSNR для значений 0..255; inf, если изображения совпадают."""
    mse = np.mean((to_arr(a) - to_arr(b)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def max_abs_diff(a, b):
    return int(np.max(np.abs(to_arr(a) - to_arr(b))))


def jpeg_roundtrip(img, quality=90):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB"), buf.getvalue()


def classify(psnr, tol_db=40.0):
    """Классификация пары по допуску, заданному до эксперимента."""
    return "identical" if psnr == float("inf") else ("close" if psnr >= tol_db else "different")


# Шесть условий сравнения (фактор варианта). Каждое условие даёт пару изображений A и B.
CONDITIONS = ["repeat", "seed_plus_1", "generator_device", "dtype", "file_format", "batch"]


def make_pair(pipe, prompt, seed, cond):
    if cond == "repeat":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed)[0]
    elif cond == "seed_plus_1":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed + 1)[0]
    elif cond == "generator_device":
        a = generate(pipe, prompt, seed, gen_device="cpu")[0]; b = generate(pipe, prompt, seed, gen_device="cuda")[0]
    elif cond == "dtype":
        pipe.to(torch.float16); a = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float32); b = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float16)
    elif cond == "file_format":
        a = generate(pipe, prompt, seed)[0]; b, _ = jpeg_roundtrip(a, 90)
    elif cond == "batch":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed, batch=4)[0]
    else:
        raise ValueError(cond)
    return a, b


def variant_summary(pipe, n):
    """Вариант n = 1..30: контекст c = (n-1)//6 + 1, условие k = (n-1)%6 + 1."""
    c, k = (n - 1) // 6, (n - 1) % 6
    seed = 1000 + n
    t0 = time.perf_counter()
    a, b = make_pair(pipe, BRIEFS[c], seed, CONDITIONS[k])
    dt = time.perf_counter() - t0
    p = psnr_db(a, b)
    pf = None
    if CONDITIONS[k] == "dtype":  # проверка причины: одинаковый начальный шум в обоих режимах
        pipe.to(torch.float16); fa = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float32); fb = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float16); pf = psnr_db(fa, fb)
    return dict(variant=n, ctx=c + 1, cond=CONDITIONS[k], seed=seed, sha_a=sha(png_bytes(a)), sha_b=sha(png_bytes(b)),
                pixel_equal=bool(np.array_equal(to_arr(a), to_arr(b))), max_diff=max_abs_diff(a, b), psnr=p,
                verdict=classify(p), seconds=round(dt, 2), psnr_fixed_noise=pf)


Ячейка 2 — `lab06_lib.py`:

In [ ]:
%%writefile lab06_lib.py
# ЛР 6. Исследование параметров text-to-image генерации. Библиотека учебных функций.
# Требуется файл lab05_lib.py из ЛР 5 в той же папке (из него берутся модель, генерация и брифы).
# Проверено 19.09.2026: Google Colab, GPU T4, Python 3.13, torch 2.11.0+cu128, diffusers 0.40.0, transformers 5.16.1.
import time
import numpy as np
import torch
from scipy import ndimage
from transformers import CLIPModel, CLIPProcessor
import lab05_lib as L

CLIP_ID = "openai/clip-vit-base-patch32"
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
SEEDS = range(101, 106)

# Короткие запросы (только предмет) и русские переводы брифов из ЛР 5.
SHORT = ["robotic arm poster", "podcast cover about the city", "science exhibition poster",
         "ecology lecture visual, forest", "electronic music cover art"]
RU = [
    "плакат-концепт для фестиваля робототехники, светящийся силуэт механической руки, глубокий синий и янтарный цвета, чистая композиция, без текста, без логотипа, без людей",
    "квадратная обложка подкаста о городской среде, абстрактный многослойный горизонт из геометрических фигур, бирюзовый и оранжевый цвета, минималистичный плоский стиль, без текста, без людей",
    "плакат-концепт научной выставки, абстрактное созвездие соединённых узлов, тёмный фон с белыми и голубыми линиями, сбалансированная композиция, без текста, без логотипа",
    "визуал для цикла лекций по экологии, стилизованный полог леса сверху, многослойные зелёные формы, мягкий утренний свет, без текста, без людей",
    "обложка электронного музыкального релиза, шлифованная металлическая поверхность с радужными бликами, стиль макросъёмки, сильный контраст, без текста, без логотипа, без людей",
]

# Шесть факторов: (имя, изменение A -> B). Базовый режим: структурированный английский запрос, 512, 1 шаг, guidance 0.
FACTORS = ["steps_1_to_4", "size_512_to_384", "size_512_to_768", "prompt_short_to_structured", "guidance_0_to_3", "language_en_to_ru"]


def load_clip():
    model = CLIPModel.from_pretrained(CLIP_ID, revision=CLIP_REVISION).to("cuda").eval()
    proc = CLIPProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
    return model, proc


@torch.no_grad()
def clip_sim(clip, img, text):
    """Косинусное сходство изображения и текста в CLIP, умноженное на 100 (не CLIPScore)."""
    model, proc = clip
    inp = proc(text=[text], images=img, return_tensors="pt", padding=True, truncation=True).to("cuda")
    out = model(**inp)
    a = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    b = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((a * b).sum()) * 100


def luma(img):
    x = np.asarray(img.convert("RGB"), dtype=np.float64) / 255
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def edge_energy(img):
    """Средняя длина градиента яркости (оператор Собеля, нормировка 1/8), как в ЛР 4."""
    y = luma(img)
    return float(np.mean(np.hypot(ndimage.sobel(y, axis=0), ndimage.sobel(y, axis=1)) / 8))


def contrast(img):
    return float(np.std(luma(img)))


def settings(factor, c):
    """Пара режимов (A, B) для контекста c = 0..4: словарь параметров generate() плюс текст запроса."""
    base = dict(prompt=L.BRIEFS[c], steps=1, guidance=0.0, size=512)
    a, b = dict(base), dict(base)
    if factor == "steps_1_to_4":
        b["steps"] = 4
    elif factor == "size_512_to_384":
        b["size"] = 384
    elif factor == "size_512_to_768":
        b["size"] = 768
    elif factor == "prompt_short_to_structured":
        a["prompt"] = SHORT[c]
    elif factor == "guidance_0_to_3":
        b["guidance"] = 3.0
    elif factor == "language_en_to_ru":
        b["prompt"] = RU[c]
    else:
        raise ValueError(factor)
    return a, b


def measure(pipe, clip, cfg, c, seed):
    """Одно изображение: метрики и время генерации, с; сходство CLIP считается с исходным английским брифом."""
    t0 = time.perf_counter()
    img = L.generate(pipe, cfg["prompt"], seed, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])[0]
    dt = time.perf_counter() - t0
    return dict(clip=clip_sim(clip, img, L.BRIEFS[c]), edge=edge_energy(img), contrast=contrast(img), seconds=dt), img


def variant_summary(pipe, clip, n, seeds=SEEDS):
    """Вариант n = 1..30: контекст c = (n-1)//6, фактор k = (n-1)%6. Для каждой метрики: A, s, B, разность, |d|/s, признак |d| > 2s."""
    c, k = (n - 1) // 6, (n - 1) % 6
    a_cfg, b_cfg = settings(FACTORS[k], c)
    A = {m: [] for m in ("clip", "edge", "contrast", "seconds")}
    B = {m: [] for m in A}
    for cfg in (a_cfg, b_cfg):  # прогрев: первый вызов с новым размером и числом шагов медленнее
        L.generate(pipe, cfg["prompt"], 0, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])
    for s in seeds:
        ma, _ = measure(pipe, clip, a_cfg, c, s)
        mb, _ = measure(pipe, clip, b_cfg, c, s)
        for m in A:
            A[m].append(ma[m]); B[m].append(mb[m])
    rows = []
    for m in A:
        d = np.mean(B[m]) - np.mean(A[m]); sa = np.std(A[m], ddof=1)
        rows.append((m, float(np.mean(A[m])), float(sa), float(np.mean(B[m])), float(d), float(abs(d) / sa) if sa > 0 else float("inf"), bool(abs(d) > 2 * sa)))
    return dict(variant=n, ctx=c + 1, factor=FACTORS[k], rows=rows)


Контроль целостности: размер и SHA-256 записанных файлов сверяются с данными раздела 6 указаний — `lab05_lib.py` 6447 байт, SHA-256 начинается с `4eb17650`; `lab06_lib.py` 6580 байт, SHA-256 начинается с `c6d8473c`. Совпадение означает, что в Engee работает ровно тот код, что проверял преподаватель.

In [ ]:
import hashlib

EXPECTED = {"lab05_lib.py": (6447, "4eb17650"), "lab06_lib.py": (6580, "c6d8473c")}
LIB_SHA = {}
for name, (size, prefix) in EXPECTED.items():
    data = (WORK / name).read_bytes()
    h = hashlib.sha256(data).hexdigest()
    LIB_SHA[name] = dict(bytes=len(data), sha256=h, matches_methodical=(len(data) == size and h.startswith(prefix)))
    print(f"{name}: {len(data)} байт, SHA-256 {h}")
    print(f"   ожидалось {size} байт, {prefix}… -> {'совпадает' if LIB_SHA[name]['matches_methodical'] else 'НЕ СОВПАДАЕТ'}")
assert all(v["matches_methodical"] for v in LIB_SHA.values()), "текст библиотек отличается от выданного"


## 3. Проверка устройства; запасной путь без CUDA

**Основной путь — GPU.** Библиотеки рассчитаны на GPU: в них жёстко записано устройство `"cuda"` (`load_pipe`, `generate_fixed_noise`, `load_clip`, `clip_sim`), а `load_pipe` по умолчанию ведёт вычисления в `float16`. В Engee GPU есть, поэтому ячейка ниже проверяет `torch.cuda.is_available()` и импортирует `lab05_lib` и `lab06_lib` **как есть**, без каких-либо замен — условия совпадают с указаниями (устройство `cuda`, `float16`, генератор шума на CPU), а в паспорте записывается модель GPU.

**Запасной путь — только если CUDA недоступна** (например, сеанс Engee запущен без ускорителя). Тогда без изменения файлов делаются две замены в памяти:

* `"cuda"` → `"cpu"` — иначе `torch` упадёт на первом же `.to("cuda")`;
* тип вычислений по умолчанию `float16` → `float32`: на CPU половинная точность для свёрток и внимания поддерживается частично и работает в разы медленнее, чем `float32`. Сами веса берутся те же — файлы `fp16` закреплённой ревизии, — они лишь приводятся к `float32` при переносе на устройство.

Файлы `lab05_lib.py` и `lab06_lib.py` на диске **не меняются** (их SHA-256 проверен выше). Ячейка читает их текст, делает в памяти только перечисленные замены, печатает, что и сколько раз заменено, и исполняет результат как модули `lab05_lib` и `lab06_lib`. Модуль `lab05_lib` регистрируется в `sys.modules` до исполнения `lab06_lib`, поэтому строка `import lab05_lib as L` внутри `lab06_lib` получает именно адаптированный модуль. Все остальные функции (генерация, метрики, `settings`, `variant_summary`) работают без изменений. Начальный шум строится генератором на CPU в обоих случаях, поэтому латенты для одного seed одинаковы; на запасном пути отличаются тип вычислений и устройство, что дало бы небольшие расхождения метрик с демонстрационными значениями и другое абсолютное время. Какой путь сработал, ячейка печатает, а `results.json` хранит признак `adaptation.applied`; если сработал запасной путь, это изменение условий записывается в паспорт.

In [ ]:
import types

for name in ("lab05_lib", "lab06_lib"):
    sys.modules.pop(name, None)

CUDA = torch.cuda.is_available()
PATCHES = {
    "lab05_lib.py": [('"cuda"', '"cpu"'), ("def load_pipe(dtype=torch.float16):", "def load_pipe(dtype=torch.float32):")],
    "lab06_lib.py": [('"cuda"', '"cpu"')],
}
ADAPTATION = {"applied": not CUDA, "replacements": []}

if CUDA:
    import lab05_lib as L
    import lab06_lib as M
    print("CUDA доступна:", torch.cuda.get_device_name(0), "- библиотеки импортированы без изменений")
else:
    for fname, reps in PATCHES.items():
        path = WORK / fname
        src = path.read_text(encoding="utf-8")
        for old, new in reps:
            n = src.count(old)
            assert n > 0, f"в {fname} не найдено {old}"
            src = src.replace(old, new)
            ADAPTATION["replacements"].append(dict(file=fname, old=old, new=new, count=n))
            print(f"{fname}: {old}  ->  {new}   замен: {n}")
        mod = types.ModuleType(fname[:-3])
        mod.__file__ = str(path)
        sys.modules[mod.__name__] = mod          # до exec: lab06_lib импортирует уже адаптированный lab05_lib
        exec(compile(src, str(path), "exec"), mod.__dict__)
    import lab05_lib as L
    import lab06_lib as M
    print("GPU нет: работают адаптированные в памяти модули, файлы на диске не изменены")

assert M.L is L
print("lab06_lib использует тот же модуль lab05_lib:", M.L is L)

## 4. Гипотеза и практический допуск (шаг 2 плана)

Записаны до запуска в `reports/journal.md` репозитория (запись внесена в репозиторий до первого выполнения этого ноутбука); здесь повторены без изменений.

**Изменяемый фактор:** только размер кадра, 512×512 (режим A) → 384×384 (режим B). Запрос, число шагов, `guidance_scale`, seed, модель и метрики одинаковы в обоих режимах.

**Механизм.** SD Turbo дистиллирована (ADD) на кадрах 512×512, карточка модели называет размер фиксированным. При 384 латент имеет 48×48 вместо 64×64 позиций: сверточная сеть и внимание работают, но на меньшей сетке, а модель за один шаг должна уложить ту же композицию в 0,56 исходной площади. Ожидается упрощение сцены — меньше мелких деталей, местами более «мыльная» текстура; композиция для того же seed может сместиться, так как сам начальный шум другого размера.

**Гипотеза по метрикам.**

| Метрика | Ожидание | Почему |
| --- | --- | --- |
| Сходство CLIP | снизится не более чем на 1,0, по критерию 2s незначимо | CLIP приводит любое изображение к 224×224, поэтому разрешение само по себе почти не влияет; главный объект и палитра должны сохраниться |
| Энергия границ (исходный размер) | **вырастет** более чем на 10 % | метрика — средний градиент на пиксель; та же сцена на меньшей сетке имеет более крутые перепады (до ×512/384 ≈ 1,33), упрощение деталей частично это компенсирует |
| Энергия границ (на одном масштабе 384) | не вырастет: в допуске или ниже | после приведения к одному масштабу остаётся только эффект «меньше деталей» |
| Контраст | в пределах ±10 %, по 2s незначимо | σ яркости от масштаба не зависит, меняется лишь из-за композиции; разброс между seed большой |
| Время генерации | **снизится на 35–50 %**, значимо по 2s | стоимость U-Net и декодера VAE пропорциональна числу пикселей (0,56), внимание — сверхлинейно; часть времени — постоянные накладные расходы (кодирование текста, запуск ядер), поэтому выигрыш может быть меньше пропорционального, но больше допуска 20 % |

**Практический допуск** (как в образце указаний, обоснован в журнале): CLIP ±1,0 (абсолютное изменение); энергия границ и контраст ±10 % от значения режима A; время ±20 % от значения режима A.

**Классификация отклика.** Цель варианта — черновик, по которому можно судить о чистовом кадре 512. Поэтому: для CLIP рост больше +1,0 — «улучшение», падение больше −1,0 — «ухудшение»; для времени снижение больше 20 % — «улучшение», рост — «ухудшение»; для энергии границ и контраста выход за ±10 % **в любую сторону** — «ухудшение» (черновик перестаёт представлять чистовой кадр), внутри допуска — «нейтрально». Признак |Δ| > 2s отвечает отдельно на вопрос «выходит ли эффект за разброс между seed».

**Ожидаемый ответ на вопрос варианта:** кадр 384 пригоден для черновиков, если CLIP и контраст останутся в допуске, время снизится больше чем на 20 %, а силуэт механической руки на всех пяти seed останется узнаваемым главным объектом.

In [ ]:
TOL = {"clip": ("abs", 1.0), "edge": ("rel", 0.10), "contrast": ("rel", 0.10), "seconds": ("rel", 0.20)}
DIRECTION = {"clip": +1, "seconds": -1, "edge": 0, "contrast": 0}   # +1 больше = лучше, -1 меньше = лучше, 0 — важна близость к A
NAMES = {"clip": "сходство CLIP", "edge": "энергия границ", "contrast": "контраст", "seconds": "время, с"}
HYPOTHESIS = {"clip": "нейтрально", "edge": "ухудшение (рост > 10 % на исходном размере)", "contrast": "нейтрально", "seconds": "улучшение"}


def verdict(metric, a, d, scale=1.0):
    kind, tol = TOL[metric]
    tol = tol * scale * (abs(a) if kind == "rel" else 1.0)
    if abs(d) <= tol:
        return "нейтрально"
    if DIRECTION[metric] == 0:
        return "ухудшение"
    return "улучшение" if d * DIRECTION[metric] > 0 else "ухудшение"


for m in TOL:
    kind, tol = TOL[m]
    print(f"{NAMES[m]:15s} допуск {'±%.1f' % tol if kind == 'abs' else '±%d %% от A' % round(tol * 100)};  ожидание: {HYPOTHESIS[m]}")

## 5. Загрузка модели и CLIP (шаг 3 плана)

`L.load_pipe()` загружает SD Turbo закреплённой ревизии (файлы `fp16`), `M.load_clip()` — CLIP ViT-B/32 закреплённой ревизии. При первом запуске время включает скачивание весов с Hugging Face (около 2,6 ГБ и 0,6 ГБ), при повторном — только чтение из кэша. Печатаются устройство и тип вычислений, чтобы паспорт фиксировал фактический режим.

In [ ]:
t0 = time.perf_counter()
pipe = L.load_pipe()
t_pipe = time.perf_counter() - t0
t0 = time.perf_counter()
clip = M.load_clip()
t_clip = time.perf_counter() - t0
LOAD = dict(pipe_seconds=round(t_pipe, 1), clip_seconds=round(t_clip, 1), unet_dtype=str(pipe.unet.dtype),
            device=str(pipe.device), clip_device=str(next(clip[0].parameters()).device), model=L.MODEL_ID,
            model_revision=L.REVISION, clip_model=M.CLIP_ID, clip_revision=M.CLIP_REVISION)
for k, v in LOAD.items():
    print(f"{k:15s} {v}")

## 6. Режимы A и B (шаг 4 плана)

Для N = 2: контекст c = (N − 1) // 6 = 0 (бриф 1), фактор k = (N − 1) % 6 = 1 (`size_512_to_384`). `M.settings` возвращает два словаря параметров; проверяем, что они различаются ровно одним полем — `size`.

In [ ]:
N = 2
c, k = (N - 1) // 6, (N - 1) % 6
a_cfg, b_cfg = M.settings(M.FACTORS[k], c)
print("Вариант", N, "| бриф", c + 1, "| фактор", M.FACTORS[k])
print("A:", a_cfg)
print("B:", b_cfg)
diff = [key for key in a_cfg if a_cfg[key] != b_cfg[key]]
print("Различаются поля:", diff)
assert diff == ["size"], "изменено не ровно одно поле"
print("Сходство CLIP считается относительно брифа:", L.BRIEFS[c] == a_cfg["prompt"])

## 7. Серия seed 101…105 и таблица метрик (шаг 5 плана)

Основной расчёт выполняет функция преподавателя `M.variant_summary(pipe, clip, 2)`: два прогревочных вызова, затем для каждого seed изображение режима A и режима B, четыре метрики и сводка по каждой метрике (A, s, B, Δ, |Δ|/s, признак |Δ| > 2s).

`variant_summary` возвращает только сводку. Чтобы сохранить значения по каждому seed и сами изображения (они нужны для сравнительного листа, проверки на одном масштабе и визуальной оценки), на время вызова функция `M.measure` оборачивается регистратором: он вызывает исходную `measure` с теми же аргументами и только запоминает её результат. Вычисления не меняются; это проверяется ниже — сводка, пересчитанная по записанным значениям, совпадает с возвращённой функцией.

In [ ]:
RECORDS = []
_measure = M.measure


def measure_logged(pipe, clip, cfg, c, seed):
    met, img = _measure(pipe, clip, cfg, c, seed)
    RECORDS.append(dict(mode="A" if cfg == a_cfg else "B", seed=seed, size=cfg["size"], img=img,
                        png_sha256=L.sha(L.png_bytes(img)), **met))
    return met, img


M.measure = measure_logged
t0 = time.perf_counter()
try:
    res = M.variant_summary(pipe, clip, N)
finally:
    M.measure = _measure
t_series = time.perf_counter() - t0
print("Вариант", res["variant"], "| бриф", res["ctx"], "| фактор", res["factor"], f"| серия заняла {t_series:.1f} с")
for m, a, s, b, d, ratio, sig in res["rows"]:
    print(f"{m:9s} A={a:.4f} s={s:.4f} B={b:.4f} d={d:+.4f} |d|/s={ratio:.2f} 2s={'да' if sig else 'нет'}")

Контроль регистратора и значения по seed. Сводка пересчитывается по записанным значениям тем же способом, что в `variant_summary` (среднее, выборочное s с `ddof=1`), и сравнивается с результатом функции. Затем — таблица по каждому seed: у одной и той же точки разброс между seed виден прямо, а не только через s.

In [ ]:
import numpy as np

MET = ("clip", "edge", "contrast", "seconds")
per = {mode: {m: [r[m] for r in RECORDS if r["mode"] == mode] for m in MET} for mode in "AB"}
assert len(RECORDS) == 10 and [r["seed"] for r in RECORDS if r["mode"] == "A"] == list(M.SEEDS)
for m, a, s, b, d, ratio, sig in res["rows"]:
    assert np.isclose(a, np.mean(per["A"][m])) and np.isclose(b, np.mean(per["B"][m])) and np.isclose(s, np.std(per["A"][m], ddof=1))
print("Сводка variant_summary совпадает с пересчётом по записанным значениям\n")

print(f"{'seed':>4s} {'реж':>3s} {'размер':>7s} {'CLIP':>8s} {'границы':>8s} {'контраст':>8s} {'время, с':>9s}  SHA-256 PNG")
for r in sorted(RECORDS, key=lambda r: (r["mode"], r["seed"])):
    print(f"{r['seed']:4d} {r['mode']:>3s} {r['size']:5d}px {r['clip']:8.3f} {r['edge']:8.4f} {r['contrast']:8.4f} {r['seconds']:9.2f}  {r['png_sha256'][:16]}…")

Итоговая таблица для отчёта: к столбцам `variant_summary` добавлены относительное изменение Δ/A, допуск и классификация по допуску (подробно — в разделе 9).

In [ ]:
ROWS = []
print(f"{'метрика':15s} {'A':>9s} {'s':>8s} {'B':>9s} {'Δ':>9s} {'Δ/A, %':>8s} {'|Δ|/s':>7s} {'|Δ|>2s':>7s}  по допуску")
for m, a, s, b, d, ratio, sig in res["rows"]:
    v = verdict(m, a, d)
    ROWS.append(dict(metric=m, A=a, s=s, B=b, delta=d, delta_rel_pct=100 * d / a, ratio=ratio, sig_2s=sig, verdict=v))
    print(f"{NAMES[m]:15s} {a:9.4f} {s:8.4f} {b:9.4f} {d:+9.4f} {100 * d / a:+8.1f} {ratio:7.2f} {'да' if sig else 'нет':>7s}  {v}")

## 8. Сравнительный лист для seed 101 (шаг 6 плана)

Код из раздела 6.2 указаний без изменений по существу (номер варианта N = 2, файл сохраняется в `artifacts/`): изображения режимов A и B для seed 101 генерируются заново, приводятся к 384×384 и ставятся рядом — слева A (512, уменьшено до 384), справа B (384). Дополнительно проверяется воспроизводимость: SHA-256 заново сгенерированных изображений сравнивается с изображениями того же seed из серии.

In [ ]:
from PIL import Image
from IPython.display import display

c, k = (N - 1) // 6, (N - 1) % 6
a_cfg, b_cfg = M.settings(M.FACTORS[k], c)
full = [L.generate(pipe, cfg["prompt"], 101, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])[0] for cfg in (a_cfg, b_cfg)]
pair = [im.resize((384, 384)) for im in full]
sheet = Image.new("RGB", (768, 384))
sheet.paste(pair[0], (0, 0)); sheet.paste(pair[1], (384, 0))
sheet.save(ART / "comparison.png")

rec101 = {r["mode"]: r for r in RECORDS if r["seed"] == 101}
REPRO_101 = {mode: L.sha(L.png_bytes(im)) == rec101[mode]["png_sha256"] for mode, im in zip("AB", full)}
print("comparison.png сохранён:", ART / "comparison.png")
print("Повторная генерация seed 101 совпала с серией бит в бит:", REPRO_101)
display(sheet)

Для визуальной оценки варианта («силуэт механической руки остаётся узнаваемым и главным объектом кадра») одного seed мало, поэтому строится лист всей серии: верхний ряд — режим A (512, уменьшено до 256), нижний — режим B (384, уменьшено до 256), столбцы — seed 101…105. Используются изображения, уже полученные в серии, новых генераций нет.

In [ ]:
T = 256
grid = Image.new("RGB", (T * 5, T * 2), "white")
for r in RECORDS:
    grid.paste(r["img"].resize((T, T), Image.LANCZOS), ((r["seed"] - 101) * T, 0 if r["mode"] == "A" else T))
grid.save(ART / "series_seeds.png")
print("series_seeds.png: верхний ряд A (512), нижний ряд B (384), seed 101…105 слева направо")
display(grid)

### Метрики на одном масштабе (требование варианта 2)

В варианте 2 указано: метрики зависят от размера, сравнивать нужно после приведения к одному масштабу. Энергия границ — средний градиент **на пиксель**, поэтому одна и та же картинка на сетке 384 даёт больший градиент, чем на 512, только из-за масштаба. Чтобы отделить этот эффект от изменения содержания, энергия границ и контраст пересчитываются функциями `M.edge_energy` и `M.contrast` в двух вариантах:

* **оба при 384:** изображение A уменьшается до 384×384 (Lanczos), B берётся как есть;
* **оба при 512:** изображение B увеличивается до 512×512 (Lanczos), A берётся как есть.

Уменьшение A ничего не выдумывает, поэтому основным считается сравнение при 384; сравнение при 512 — контрольное (увеличение B сглаживает границы и занижает его энергию). Сходство CLIP не пересчитывается: процессор CLIP сам приводит любое изображение к 224×224. Дополнительно — «чистый» эффект масштаба: энергия границ A при 512 и того же A, уменьшенного до 384.

In [ ]:
def summarize(a_vals, b_vals):
    a_vals, b_vals = np.asarray(a_vals), np.asarray(b_vals)
    a, b, s = a_vals.mean(), b_vals.mean(), a_vals.std(ddof=1)
    d = b - a
    return dict(A=float(a), s=float(s), B=float(b), delta=float(d), delta_rel_pct=float(100 * d / a), ratio=float(abs(d) / s), sig_2s=bool(abs(d) > 2 * s))


imgs = {mode: [r["img"] for r in sorted(RECORDS, key=lambda r: r["seed"]) if r["mode"] == mode] for mode in "AB"}
A384 = [im.resize((384, 384), Image.LANCZOS) for im in imgs["A"]]
B512 = [im.resize((512, 512), Image.LANCZOS) for im in imgs["B"]]
COMMON = {}
for metric, fn in (("edge", M.edge_energy), ("contrast", M.contrast)):
    COMMON[f"{metric}@384"] = summarize([fn(x) for x in A384], [fn(x) for x in imgs["B"]])
    COMMON[f"{metric}@512"] = summarize([fn(x) for x in imgs["A"]], [fn(x) for x in B512])
scale_only = summarize([M.edge_energy(x) for x in imgs["A"]], [M.edge_energy(x) for x in A384])
COMMON["edge: только масштаб (A 512 -> A 384)"] = scale_only

print(f"{'сравнение':40s} {'A':>8s} {'s':>8s} {'B':>8s} {'Δ':>9s} {'Δ/A, %':>8s} {'|Δ|/s':>7s} {'2s':>4s}  по допуску")
for name, r in COMMON.items():
    v = verdict(name.split("@")[0], r["A"], r["delta"]) if "@" in name else "—"
    r["verdict"] = v
    print(f"{name:40s} {r['A']:8.4f} {r['s']:8.4f} {r['B']:8.4f} {r['delta']:+9.4f} {r['delta_rel_pct']:+8.1f} {r['ratio']:7.2f} {'да' if r['sig_2s'] else 'нет':>4s}  {v}")

## 9. Классификация по допуску и сравнение с критерием 2s (шаг 7 плана)

Для каждой метрики сопоставляются три ответа: ожидание из гипотезы, признак |Δ| > 2s (выходит ли эффект за разброс между seed) и классификация по практическому допуску. Отдельно отмечены метрики, где критерий 2s и допуск дают **разный** ответ: «значимо, но в допуске» (статистически заметно, практически неважно) или «вне допуска, но не значимо» (большой эффект на фоне большого разброса — пяти seed мало, чтобы его подтвердить).

In [ ]:
DISAGREE = []
print(f"{'метрика':15s} {'гипотеза':28s} {'|Δ|>2s':>7s} {'допуск':>12s}  согласие 2s и допуска")
for r in ROWS:
    out_tol = r["verdict"] != "нейтрально"
    agree = out_tol == r["sig_2s"]
    note = "согласны" if agree else ("значимо по 2s, но в допуске" if r["sig_2s"] else "вне допуска, но не значимо по 2s")
    if not agree:
        DISAGREE.append(dict(metric=r["metric"], note=note))
    r["agree_2s_tol"] = agree
    print(f"{NAMES[r['metric']]:15s} {HYPOTHESIS[r['metric']]:28.28s} {'да' if r['sig_2s'] else 'нет':>7s} {r['verdict']:>12s}  {note}")
print("\nРасхождения критерия 2s и допуска:", DISAGREE if DISAGREE else "нет")

**Чувствительность вывода к допуску** (контрольный вопрос 9 и продвинутый уровень рубрики). Классификация повторяется при допуске, уменьшенном вдвое и увеличенном вдвое, и для каждой метрики вычисляется пороговое значение допуска, при котором ответ меняется (это |Δ| в единицах допуска: абсолютное для CLIP, относительное для остальных). Если порог далеко от выбранного допуска, вывод устойчив к его выбору.

In [ ]:
SENS = {}
print(f"{'метрика':15s} {'допуск ×0,5':>13s} {'×1':>13s} {'×2':>13s}  порог смены ответа")
for r in ROWS:
    m = r["metric"]
    kind, tol = TOL[m]
    vs = {x: verdict(m, r["A"], r["delta"], x) for x in (0.5, 1.0, 2.0)}
    flip = abs(r["delta"]) if kind == "abs" else abs(r["delta"] / r["A"])
    SENS[m] = dict(half=vs[0.5], base=vs[1.0], double=vs[2.0], flip_threshold=flip, flip_in_units_of_tol=flip / tol)
    thr = f"{flip:.2f} (CLIP)" if kind == "abs" else f"{100 * flip:.1f} %"
    print(f"{NAMES[m]:15s} {vs[0.5]:>13s} {vs[1.0]:>13s} {vs[2.0]:>13s}  {thr} = {flip / tol:.2f} допуска")

## 10. Намеренная ошибка демонстрационного варианта (шаг 8 плана)

Код раздела 6.2 указаний без изменений. **Симптом, который проверяется:** ожидание, что `guidance_scale = 1,0` «включает» управление запросом у SD Turbo. В Diffusers управление без классификатора включается только при `guidance_scale > 1`, поэтому изображение при 1,0 должно совпасть с изображением при 0,0 бит в бит, а `pipe.do_classifier_free_guidance` — быть `False`; при 3,0 признак становится `True`, и изображение заметно меняется (в указаниях PSNR 11,6 дБ). Генерация при 3,0 вдвое дороже (пакет «с текстом» и «без текста»).

In [ ]:
# Намеренная ошибка: ожидание, что guidance_scale = 1.0 «включает» guidance у SD Turbo.
p = L.BRIEFS[0]
x0 = L.generate(pipe, p, 101, guidance=0.0)[0]
x1 = L.generate(pipe, p, 101, guidance=1.0)[0]
print("guidance 1.0: результат совпал с guidance 0.0 бит в бит:", L.sha(L.png_bytes(x0)) == L.sha(L.png_bytes(x1)), "| CFG активен:", pipe.do_classifier_free_guidance)
cfg_at_1 = pipe.do_classifier_free_guidance
x3 = L.generate(pipe, p, 101, guidance=3.0)[0]
print("guidance 3.0: CFG активен:", pipe.do_classifier_free_guidance, "| PSNR относительно guidance 0.0, дБ:", round(L.psnr_db(x0, x3), 2))
GUIDANCE = dict(sha_equal_0_vs_1=L.sha(L.png_bytes(x0)) == L.sha(L.png_bytes(x1)), cfg_at_1=cfg_at_1,
                cfg_at_3=pipe.do_classifier_free_guidance, psnr_0_vs_3_db=round(L.psnr_db(x0, x3), 2),
                sha_x0=L.sha(L.png_bytes(x0)), methodical=dict(sha_equal=True, cfg_at_1=False, cfg_at_3=True, psnr_db=11.6))
GUIDANCE["x0_equals_series_A_seed101"] = GUIDANCE["sha_x0"] == rec101["A"]["png_sha256"]
print("x0 (бриф 1, seed 101, 512, guidance 0) совпадает с изображением A seed 101 из серии:", GUIDANCE["x0_equals_series_A_seed101"])

## 11. Воспроизведение демонстрационного варианта 1 и сопоставление

Продвинутый уровень блока «Артефакт и воспроизводимость» требует воспроизвести результат демонстрационного варианта и сопоставить с указаниями. Вариант 1 — тот же бриф 1, фактор «число шагов 1 → 4». Запуск той же функцией `M.variant_summary(pipe, clip, 1)`; значения из таблицы раздела 6.3 указаний (Colab, T4, `float16`) записаны ниже вручную.

Работа идёт на GPU в `float16`, как в указаниях, поэтому значения CLIP, энергии границ и контраста должны совпасть с таблицей указаний с точностью до различий модели GPU и версий torch/CUDA/diffusers (на T4 с той же конфигурацией в указаниях они совпали в трёх сеансах); выводы должны совпасть полностью. Сопоставляются (а) значения CLIP, энергии границ и контраста — относительное расхождение с указаниями; (б) выводы — признак 2s и классификация по допуску указаний (CLIP ±1,0; границы и контраст ±10 %; время ±20 %); (в) время — не абсолютное (другое устройство), а отношение B/A. Для демо-варианта ответ по допуску записывается как «в допуске / вне допуска», как в разделе 6.3 указаний: направления «улучшение/ухудшение» в разделе 4 заданы для задачи варианта 2. На GPU шаг занимает несколько секунд (на запасном пути CPU — 5–10 мин).

In [ ]:
DEMO_REF = {  # раздел 6.3 указаний: A, s, B, Δ, |Δ|/s, |Δ| > 2s
    "clip": (30.796, 1.394, 30.292, -0.504, 0.36, False),
    "edge": (0.0457, 0.0069, 0.0920, +0.0463, 6.68, True),
    "contrast": (0.1923, 0.0226, 0.2778, +0.0855, 3.79, True),
    "seconds": (0.2854, 0.0027, 0.5266, +0.2412, 87.77, True),
}
t0 = time.perf_counter()
res1 = M.variant_summary(pipe, clip, 1)
t_demo = time.perf_counter() - t0
print("Вариант", res1["variant"], "фактор", res1["factor"], f"| {t_demo:.1f} с")
for m, a, s, b, d, ratio, sig in res1["rows"]:
    print(f"{m:9s} A={a:.4f} s={s:.4f} B={b:.4f} d={d:+.4f} |d|/s={ratio:.2f} 2s={'да' if sig else 'нет'}")

DEMO_CMP = []
print(f"\n{'метрика':15s} {'A указ.':>8s} {'A здесь':>8s} {'расх. A':>8s} {'B указ.':>8s} {'B здесь':>8s} {'расх. B':>8s}  2s указ./здесь  допуск указ./здесь")
for m, a, s, b, d, ratio, sig in res1["rows"]:
    ra, rs, rb, rd, rr, rsig = DEMO_REF[m]
    v_ref, v_here = [("в допуске" if verdict(m, x, y) == "нейтрально" else "вне допуска") for x, y in ((ra, rd), (a, d))]
    row = dict(metric=m, A_ref=ra, A=a, B_ref=rb, B=b, sig_ref=rsig, sig=sig, verdict_ref=v_ref, verdict=v_here)
    if m == "seconds":
        row.update(ratio_BA_ref=rb / ra, ratio_BA=b / a)
        dev_a, dev_b = f"B/A {rb / ra:.2f}", f"B/A {b / a:.2f}"
    else:
        row.update(dev_A_pct=100 * (a - ra) / ra, dev_B_pct=100 * (b - rb) / rb)
        dev_a, dev_b = f"{row['dev_A_pct']:+.1f} %", f"{row['dev_B_pct']:+.1f} %"
    row["conclusion_match"] = (rsig == sig) and (v_ref == v_here)
    DEMO_CMP.append(row)
    print(f"{NAMES[m]:15s} {ra:8.4f} {a:8.4f} {dev_a:>8s} {rb:8.4f} {b:8.4f} {dev_b:>8s}  {'да' if rsig else 'нет':>5s}/{'да' if sig else 'нет':<4s}  {v_ref:>11s}/{v_here}")
print("\nВыводы демонстрационного варианта воспроизведены по всем метрикам:", all(r["conclusion_match"] for r in DEMO_CMP))

## 12. Сохранение результатов

В `artifacts/results.json` записывается всё, что нужно для отчёта и паспорта: среда, адаптация, контрольные суммы библиотек, время загрузки, режимы A и B, сводка `variant_summary`, значения по каждому seed с SHA-256 изображений, сравнение на одном масштабе, классификация, чувствительность к допуску, проверка guidance и сопоставление с демонстрационным вариантом. Изображения в JSON не входят — они сохранены PNG-файлами.

In [ ]:
import json

RESULTS = dict(
    lab="ЛР 6. Исследование параметров text-to-image генерации", author="Яковенко Максим Михайлович", variant=N,
    brief=1, factor=M.FACTORS[k], seeds=list(M.SEEDS), work_dir=str(WORK), environment=ENV, adaptation=ADAPTATION,
    libraries=LIB_SHA, load=LOAD, settings=dict(A=a_cfg, B=b_cfg), brief_text=L.BRIEFS[c],
    tolerance={m: dict(kind=v[0], value=v[1]) for m, v in TOL.items()}, direction=DIRECTION, hypothesis=HYPOTHESIS,
    summary=ROWS, series_seconds=round(t_series, 1),
    per_seed=[{key: val for key, val in r.items() if key != "img"} for r in sorted(RECORDS, key=lambda r: (r["mode"], r["seed"]))],
    common_scale=COMMON, disagreement_2s_vs_tolerance=DISAGREE, tolerance_sensitivity=SENS,
    comparison_seed101_reproduced=REPRO_101, guidance_check=GUIDANCE,
    demo_variant1=dict(rows=[list(r) for r in res1["rows"]], seconds_total=round(t_demo, 1), comparison=DEMO_CMP),
)
(ART / "results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")
print("results.json:", (ART / "results.json").stat().st_size, "байт")

## 13. Контрольные суммы артефактов

SHA-256 всех файлов из `artifacts/` и библиотек: по ним проверяется, что файлы, скачанные из Engee в репозиторий, — те самые, что получены в этом запуске. Последней строкой — общее время выполнения ноутбука.

In [ ]:
for path in sorted(ART.iterdir()) + [WORK / "lab05_lib.py", WORK / "lab06_lib.py"]:
    if path.is_file():
        print(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.stat().st_size:9d}  {path.relative_to(WORK)}")
print(f"\nОбщее время выполнения ноутбука: {(time.perf_counter() - T_START) / 60:.1f} мин")

## 14. Визуальная оценка сравнительного листа

**ЗАПОЛНЯЕТСЯ ПОСЛЕ ВЫПОЛНЕНИЯ**

## 15. Вывод

**ЗАПОЛНЯЕТСЯ ПОСЛЕ ВЫПОЛНЕНИЯ**

## 16. Ограничения

**ЗАПОЛНЯЕТСЯ ПОСЛЕ ВЫПОЛНЕНИЯ**

## 17. Паспорт эксперимента

**ЗАПОЛНЯЕТСЯ ПОСЛЕ ВЫПОЛНЕНИЯ**